# DREAM: Diffusion-based PMW Rainfall rEtrievals with Attention Mechanisms

<p style="color: red;"><strong>Note: The same ResUNet predictions are used for all diffusion models compared in this study. ResUNet is trained without SWD regularization. This paper focuses on improving the diffusion model and the final rainfall retrievals obtained by correcting the ResUNet predictions.</strong></p>

**Defaults:** 50,000 requested patchesfor training; 25 epochs; training batch 32; 1,000 diffusion steps; AdamW learning rate 1e-4; MSE weight 0.55; rainfall SWD weight 0.45; 32 random projections; one sampling realization.

**Loss:** Noise MSE uses independent full-range timesteps. Rainfall SWD uses a separate shared mini-batch timestep in indices 0–196 and its own forward-noise realization. Each patch is a 1,024-dimensional observation after applying the same DPR > 0.2 mask to prediction and target; projected patch values are sorted across the batch. This is image-level SWD, not a pooled rainy-pixel distribution loss.

**Selection:** The executable source requires at least 10% nonzero residual pixels, despite conflicting source comments. Half of the requested samples are selected from each mean-LSM group when available. Both training and validation inherit this selection. There is no held-out test set here, and random patch splitting does not enforce orbit/year independence.

**Corrections made:** Fit scalers on training data only; restore saved scalers for checkpoint sampling; limit visualization count to available patches; use one shared difference-map color scale. Signed residuals, training rainfall estimates, and sampled outputs remain unclipped. Rain map colors start at zero, so negative output values are assessed through diagnostics and signed arrays.

Data loading and normalized copies require substantial RAM. Start with a smaller sample count and one epoch to check the workflow.

## 1. Dependencies

PyTorch must be available in the runtime. Install other packages only if missing.

In [ ]:
import importlib.util
import subprocess
import sys
packages = {'numpy': 'numpy', 'matplotlib': 'matplotlib', 'tqdm': 'tqdm', 'sklearn': 'scikit-learn'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Install PyTorch for this runtime first.')

## 2. Imports and seed

The seed makes selection, model initialization, and splitting reproducible. Sampling retains the supplied member seeds.

In [ ]:
import os
import copy
import time
import json
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

## 3. Optional Drive mount

Enable this to read patches and save outputs on Drive.

In [ ]:
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## 4. Paths and settings

Edit this cell before running. For a workflow check, use NUM_SAMPLES=100, NUM_EPOCHS=1, and fewer visualization patches. SAMPLE_ONLY skips optimization and requires a trusted existing checkpoint; it still loads and splits the dataset.

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/dream_data' if USE_GOOGLE_DRIVE else '/content/dream_data')
PATCHES_DIR = DATA_ROOT / 'grl_dpr_patches_with_resunet_output_and_residue'
OUTPUT_DIR = Path('/content/drive/MyDrive/dream_outputs' if USE_GOOGLE_DRIVE else '/content/dream_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_PATH = OUTPUT_DIR / 'best_diffusion_model_rainfall_minibatch_DPR_rainy_SWD.pth'
NUM_SAMPLES = 55000
NUM_EPOCHS = 30
TRAIN_FRACTION = 0.9
TRAIN_BATCH_SIZE = 32
VAL_BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
DIFFUSION_STEPS = 1000
SWD_TIMESTEP_UPPER = 197  # torch.randint upper bound is exclusive: 0–196.
NUM_PROJECTIONS = 32
ZERO_THRESHOLD = 0.2
MSE_WEIGHT = 0.55
RAIN_SWD_WEIGHT = 0.45
NUM_REALIZATIONS = 1
NUM_VISUALIZATION_SAMPLES = 5
MIN_NONZERO_RESIDUE_FRACTION = 0.1
SAMPLE_ONLY = False
# Names used in the supplied loop and visualization code.
zero_threshold = ZERO_THRESHOLD
mse_weight = MSE_WEIGHT
rain_swd_weight = RAIN_SWD_WEIGHT
num_realizations = NUM_REALIZATIONS
num_visualization_samples = NUM_VISUALIZATION_SAMPLES
best_checkpoint_path = str(BEST_CHECKPOINT_PATH)
if NUM_SAMPLES < 2 or NUM_SAMPLES % 2:
    raise ValueError('NUM_SAMPLES must be an even integer of at least 2 for equal LSM quotas.')
if not 0 < TRAIN_FRACTION < 1:
    raise ValueError('TRAIN_FRACTION must be between zero and one.')
if not 0 < SWD_TIMESTEP_UPPER <= DIFFUSION_STEPS:
    raise ValueError('SWD timestep range must fit within the diffusion schedule.')
if min(TRAIN_BATCH_SIZE, VAL_BATCH_SIZE, NUM_REALIZATIONS, NUM_VISUALIZATION_SAMPLES, NUM_PROJECTIONS) < 1:
    raise ValueError('Batch sizes, projections, realizations, and visualization count must be positive.')
if NUM_EPOCHS < 1 and not SAMPLE_ONLY:
    raise ValueError('Training requires at least one epoch.')

## 5. Data processor

Standardize TB and ERA5. Residuals pass through unchanged; their statistics are diagnostic only.

In [ ]:
class SimpleDataProcessor:
    """Simple data processor for signed residue training."""
    def __init__(self, zero_threshold=0.2):
        self.tb_scalers = [StandardScaler() for _ in range(13)]
        self.era5_scalers = [StandardScaler() for _ in range(9)]
        self.rain_stats = {}
        self.zero_threshold = zero_threshold

    def fit(self, tb_data, era5_data, rain_data):
        """Fit scalers on training data. Here rain_data is the signed residue target."""
        print("Fitting scalers...")

        for i in range(13):
            tb_channel = tb_data[..., i].flatten()
            tb_channel = tb_channel[~np.isnan(tb_channel)]
            self.tb_scalers[i].fit(tb_channel.reshape(-1, 1))

        for i in range(9):
            era5_channel = era5_data[..., i].flatten()
            era5_channel = era5_channel[~np.isnan(era5_channel)]
            self.era5_scalers[i].fit(era5_channel.reshape(-1, 1))

        # Signed residue statistics (NO log1p, NO zero-threshold clipping)
        rain_flat = rain_data.flatten()
        rain_flat = rain_flat[~np.isnan(rain_flat)]
        rain_flat = np.nan_to_num(rain_flat, nan=0.0, posinf=0.0, neginf=0.0)

        mean_val = np.mean(rain_flat)
        std_val = max(np.std(rain_flat), 1e-6)
        p1 = np.percentile(rain_flat, 1) if len(rain_flat) > 0 else 0
        p99 = np.percentile(rain_flat, 99) if len(rain_flat) > 0 else 0

        self.rain_stats = {
            'mean': mean_val,
            'std': std_val,
            'min': rain_flat.min() if len(rain_flat) > 0 else 0,
            'max': rain_flat.max() if len(rain_flat) > 0 else 0,
            'p1': p1,
            'p99': p99,
        }

        print(f"Signed Residue Statistics:")
        print(f"  Mean: {self.rain_stats['mean']:.4f}")
        print(f"  Std : {self.rain_stats['std']:.4f}")
        print(f"  Min : {self.rain_stats['min']:.4f}")
        print(f"  Max : {self.rain_stats['max']:.4f}")
        print(f"  p1  : {self.rain_stats['p1']:.4f}")
        print(f"  p99 : {self.rain_stats['p99']:.4f}")
        print(f"  IMPORTANT: No 0.2-threshold clipping is applied for residue")

        return None, None

    def transform_tb(self, tb_data):
        """Transform TB features"""
        transformed = np.zeros_like(tb_data)
        for i in range(13):
            channel_data = tb_data[..., i]
            original_shape = channel_data.shape
            channel_flat = channel_data.reshape(-1, 1)
            channel_transformed = self.tb_scalers[i].transform(channel_flat)
            transformed[..., i] = channel_transformed.reshape(original_shape)
        return transformed

    def transform_era5(self, era5_data):
        """Transform ERA5 features"""
        transformed = np.zeros_like(era5_data)
        for i in range(9):
            channel_data = era5_data[..., i]
            original_shape = channel_data.shape
            channel_flat = channel_data.reshape(-1, 1)
            channel_transformed = self.era5_scalers[i].transform(channel_flat)
            transformed[..., i] = channel_transformed.reshape(original_shape)
        return transformed

    def transform_rain_to_normalized(self, rain_data, device='cpu'):
        """No normalization for residue; return residue as-is."""
        if torch.is_tensor(rain_data):
            return rain_data.to(device)
        return torch.FloatTensor(rain_data).to(device)

    def transform_rain_to_physical(self, rain_normalized):
        """No inverse transform for residue; return residue as-is."""
        if torch.is_tensor(rain_normalized):
            return rain_normalized
        return torch.FloatTensor(rain_normalized)

    def apply_zero_threshold(self, rain_physical):
        """No-op for residue."""
        return rain_physical

## 6. Timestep embedding

Encode each diffusion timestep as sine/cosine features.

In [ ]:
def timestep_embedding(timesteps, dim, max_period=10000, device=None):
    """Create sinusoidal timestep embeddings"""
    if device is None:
        device = timesteps.device
    
    half = dim // 2
    max_period_tensor = torch.tensor(max_period, dtype=torch.float32, device=device)
    
    freqs = torch.exp(
        -torch.log(max_period_tensor) * 
        torch.arange(half, dtype=torch.float32, device=device) / half
    )
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding

## 7. TB encoder

Extract conditioning features at 16, 8, 4, and 2 pixels per side.

In [ ]:
class TBEncoder(nn.Module):
    """TB Encoder: Extracts multi-scale TB features with proper spatial alignment"""
    def __init__(self):
        super().__init__()
        
        # Encoder layers for TB features with proper spatial dimensions
        # Level 1: 32x32 → 16x16
        self.conv1 = nn.Sequential(
            nn.Conv2d(13, 64, 3, stride=2, padding=1),  # 32→16
            nn.GroupNorm(8, 64),
            nn.SiLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.GroupNorm(8, 64),
            nn.SiLU()
        )
        
        # Level 2: 16x16 → 8x8
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # 16→8
            nn.GroupNorm(8, 128),
            nn.SiLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.GroupNorm(8, 128),
            nn.SiLU()
        )
        
        # Level 3: 8x8 → 4x4
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # 8→4
            nn.GroupNorm(8, 256),
            nn.SiLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.GroupNorm(8, 256),
            nn.SiLU()
        )
        
        # Level 4: 4x4 → 2x2
        self.conv4 = nn.Sequential(
            nn.Conv2d(256, 512, 3, stride=2, padding=1),  # 4→2
            nn.GroupNorm(8, 512),
            nn.SiLU(),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.GroupNorm(8, 512),
            nn.SiLU()
        )
        
    def forward(self, tb):
        # Extract multi-scale TB features with proper spatial dimensions
        tb1 = self.conv1(tb)  # [N, 64, 16, 16] - matches after first downsampling ## N is the batch size
        
        tb2 = self.conv2(tb1)  # [N, 128, 8, 8] - matches after second downsampling
        
        tb3 = self.conv3(tb2)  # [N, 256, 4, 4] - matches after third downsampling
        
        tb4 = self.conv4(tb3)  # [N, 512, 2, 2] - matches after fourth downsampling
        
        return tb1, tb2, tb3, tb4

## 8. ERA5 encoder

Map nine reanalysis inputs to the 2 × 2 bottleneck.

In [ ]:
class ERA5Processor(nn.Module):
    """ERA5 Processor: Converts ERA5 fields into compact latent representation"""
    def __init__(self):
        super().__init__()
        
        self.processor = nn.Sequential(
            # Initial processing (keep at 32x32)
            nn.Conv2d(9, 64, 3, padding=1),
            nn.GroupNorm(8, 64),
            nn.SiLU(),
            
            # Downsample to 16x16
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.GroupNorm(8, 128),
            nn.SiLU(),
            
            # Downsample to 8x8
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.GroupNorm(8, 256),
            nn.SiLU(),
            
            # Downsample to 4x4
            nn.Conv2d(256, 512, 3, stride=2, padding=1),
            nn.GroupNorm(8, 512),
            nn.SiLU(),
            
            # Downsample to 2x2
            nn.Conv2d(512, 512, 3, stride=2, padding=1),
            nn.GroupNorm(8, 512),
            nn.SiLU()
        )
        
    def forward(self, era5):
        # Process ERA5 fields to match bottleneck dimensions
        era5_latent = self.processor(era5)  # [N, 512, 2, 2]
        return era5_latent

## 9. Channel attention

Use average/max pooling and a shared channel gate.

In [ ]:
class ChannelAttention(nn.Module):
    """Channel attention matching:
       AvgPool/MaxPool -> Conv -> ReLU -> Conv -> Add -> Sigmoid -> Multiply
    """
    def __init__(self, channels, reduction_ratio=16):
        super().__init__()

        reduced_channels = max(1, channels // reduction_ratio)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        # Shared MLP implemented with 1x1 convolutions
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(channels, reduced_channels, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced_channels, channels, kernel_size=1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Two channel descriptors
        avg_out = self.shared_mlp(self.avg_pool(x))   # (B, C, 1, 1)
        max_out = self.shared_mlp(self.max_pool(x))   # (B, C, 1, 1)

        # Add, then sigmoid
        attention = self.sigmoid(avg_out + max_out)   # (B, C, 1, 1)

        # Multiply with input feature map
        out = x * attention

        return out

## 10. Spatial attention

Compute spatial weights from channel-pooled features.

In [ ]:
class SpatialAttention(nn.Module):
    """Spatial attention module"""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # Channel pooling
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        
        # Concatenate and convolve
        combined = torch.cat([avg_out, max_out], dim=1)
        attention = self.conv(combined)
        
        return self.sigmoid(attention)

## 11. Cross-attention

Inject TB information through queries, keys, and values.

In [ ]:
class CrossAttentionBlock(nn.Module):
    def __init__(self, query_dim, key_dim, hidden_dim=None, num_heads=8, dropout=0.1):
        super().__init__()
        
        if hidden_dim is None:
            hidden_dim = 4 * query_dim
        
        assert query_dim % num_heads == 0, "query_dim must be divisible by num_heads"
        
        self.norm_query = nn.LayerNorm(query_dim)
        self.norm_keyvalue = nn.LayerNorm(key_dim)
        
        # Always project for consistency (or make it configurable)
        self.key_proj = nn.Linear(key_dim, query_dim)
        self.value_proj = nn.Linear(key_dim, query_dim)
        
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=query_dim,
            num_heads=num_heads,
            batch_first=True,
            dropout=dropout
        )
        self.attn_dropout = nn.Dropout(dropout)  # Optional extra dropout
        
        self.ffn_norm = nn.LayerNorm(query_dim)
        self.ffn = nn.Sequential(
            nn.Linear(query_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, query_dim),
            nn.Dropout(dropout)
        )

    def forward(self, query, key_value):
        B, C, H, W = query.shape
        Bk, Ck, Hk, Wk = key_value.shape
        assert B == Bk, "Batch size mismatch"
        
        # Flatten spatial dimensions
        query_flat = query.flatten(2).transpose(1, 2)      # [B, HW, C]
        kv_flat = key_value.flatten(2).transpose(1, 2)     # [B, HkWk, Ck]
        
        # Pre-norm
        query_norm = self.norm_query(query_flat)
        kv_norm = self.norm_keyvalue(kv_flat)
        
        # Project key/value
        key = self.key_proj(kv_norm)
        value = self.value_proj(kv_norm)
        
        # Cross attention
        attn_out, _ = self.cross_attn(query=query_norm, key=key, value=value)
        query_flat = query_flat + self.attn_dropout(attn_out)
        
        # FFN with pre-norm
        query_flat = query_flat + self.ffn(self.ffn_norm(query_flat))
        
        # Reshape back
        output = query_flat.transpose(1, 2).reshape(B, C, H, W)
        return output

## 12. Stage-aware residual block

Channel attention is used at shallow stages and self-attention at deeper stages.

In [ ]:
class MultiStageConditionalBlock(nn.Module):
    """Residual block with stage-aware attention.

    Uses channel attention when self-attention is not used (stage_level < 3),
    and self-attention at deeper stages (stage_level >= 3).
    """
    def __init__(self, in_channels, out_channels, time_channels,
                 tb_channels=None, stage_level=1, num_heads=4):
        super().__init__()

        self.stage_level = stage_level

        norm_groups1 = min(8, in_channels)
        norm_groups2 = min(8, out_channels)

        self.conv1 = nn.Sequential(
            nn.GroupNorm(norm_groups1, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, 3, padding=1)
        )

        self.time_embed = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_channels, out_channels)
        )

        self.tb_injection = None
        if tb_channels is not None:
            self.tb_injection = nn.Sequential(
                nn.Conv2d(tb_channels, out_channels, 1),
                nn.GroupNorm(min(8, out_channels), out_channels),
                nn.SiLU()
            )

        self.conv2 = nn.Sequential(
            nn.GroupNorm(norm_groups2, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1)
        )

        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.residual_conv = nn.Identity()

        if stage_level < 3:
            self.channel_attn = ChannelAttention(out_channels)
            self.attn = None
        else:
            self.channel_attn = None
            self.attn = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)

    def forward(self, x, t_emb, tb_features=None):
        residual = self.residual_conv(x)

        h = self.conv1(x)

        t_emb = self.time_embed(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + t_emb

        if tb_features is not None and self.tb_injection is not None:
            if tb_features.shape[-2:] != h.shape[-2:]:
                tb_features = F.interpolate(tb_features, size=h.shape[-2:], mode='bilinear', align_corners=False)
            h = h + self.tb_injection(tb_features)

        h = self.conv2(h)

        if self.channel_attn is not None:
            h = self.channel_attn(h)

        if self.attn is not None:
            B, C, H, W = h.shape
            h_flat = h.flatten(2).transpose(1, 2)
            h_attn, _ = self.attn(h_flat, h_flat, h_flat)
            h = h + h_attn.transpose(1, 2).reshape(B, C, H, W)

        return h + residual

## 13. Bottleneck fusion

Fuse TB and ERA5 using bidirectional attention and channel/spatial weights.

In [ ]:
class BottleneckFusion(nn.Module):
    """Fuses TB and ERA5 features using bidirectional cross-attention."""
    def __init__(self, channels=512, num_heads=8):
        super().__init__()

        self.attention = nn.MultiheadAttention(channels, num_heads, batch_first=True)
        self.channel_attn = ChannelAttention(channels * 2)
        self.spatial_attn = SpatialAttention()

        self.fusion = nn.Sequential(
            nn.GroupNorm(8, channels * 2),
            nn.SiLU(),
            nn.Conv2d(channels * 2, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1)
        )

    def forward(self, tb_features, era5_latent):
        B, C, H, W = tb_features.shape

        if era5_latent.shape[2:] != (H, W):
            era5_latent = F.interpolate(era5_latent, size=(H, W), mode='bilinear', align_corners=False)

        tb_flat = tb_features.flatten(2).transpose(1, 2)
        era5_flat = era5_latent.flatten(2).transpose(1, 2)

        tb_attended, _ = self.attention(tb_flat, era5_flat, era5_flat)
        tb_attended = tb_attended.transpose(1, 2).reshape(B, C, H, W)

        era5_attended, _ = self.attention(era5_flat, tb_flat, tb_flat)
        era5_attended = era5_attended.transpose(1, 2).reshape(B, C, H, W)

        combined = torch.cat([tb_attended, era5_attended], dim=1)
        combined = self.channel_attn(combined)

        spatial_weights = self.spatial_attn(combined)
        combined = combined * spatial_weights

        fused = self.fusion(combined)
        return fused

## 14. Conditional diffusion network

Predict noise from the noisy signed residual, TB, ERA5, and timestep. Architecture retained as supplied.

In [ ]:
class DiffusionRainModel(nn.Module):
    """UNet for rain diffusion with innovative conditioning mechanism"""
    def __init__(self, time_dim=256):
        super().__init__()
        
        print("Initializing model with innovative conditioning mechanism...")
        
        # Time embedding
        self.time_embed = nn.Sequential(
            nn.Linear(time_dim, time_dim * 2),
            nn.SiLU(),
            nn.Linear(time_dim * 2, time_dim * 2),
            nn.SiLU(),
            nn.Linear(time_dim * 2, time_dim)
        )
        
        # ========== CONDITIONING MODULES ==========
        # TB Encoder for multi-scale feature extraction
        self.tb_encoder = TBEncoder()
        
        # ERA5 Processor for latent representation
        self.era5_processor = ERA5Processor()
        
        # Bottleneck Fusion module
        self.bottleneck_fusion = BottleneckFusion()

        # Cross-attention modules at the exact multi-scale injection locations
        self.cross_attn_16 = CrossAttentionBlock(query_dim=64, key_dim=64, num_heads=4, dropout=0.0)
        self.cross_attn_8 = CrossAttentionBlock(query_dim=128, key_dim=128, num_heads=4, dropout=0.0)
        self.cross_attn_4 = CrossAttentionBlock(query_dim=256, key_dim=256, num_heads=8, dropout=0.0)
        
        # ========== NOISY IMAGE ENCODER ==========
        # Initial projection (keep at 32x32)
        self.initial_proj = nn.Conv2d(1, 64, 3, padding=1)
        
        # Encoder blocks with multi-stage TB injection
        # Level 1: 32x32 → 16x16
        self.enc1 = MultiStageConditionalBlock(64, 64, time_dim, tb_channels=64, stage_level=1)
        self.down1 = nn.Conv2d(64, 64, 3, stride=2, padding=1)  # 32→16
        
        # Level 2: 16x16 → 8x8
        self.enc2 = MultiStageConditionalBlock(64, 128, time_dim, tb_channels=128, stage_level=2)
        self.down2 = nn.Conv2d(128, 128, 3, stride=2, padding=1)  # 16→8
        
        # Level 3: 8x8 → 4x4
        self.enc3 = MultiStageConditionalBlock(128, 256, time_dim, tb_channels=256, stage_level=3)
        self.down3 = nn.Conv2d(256, 256, 3, stride=2, padding=1)  # 8→4
        
        # Level 4: 4x4 → 2x2
        self.enc4 = MultiStageConditionalBlock(256, 512, time_dim, tb_channels=512, stage_level=4)
        self.down4 = nn.Conv2d(512, 512, 3, stride=2, padding=1)  # 4→2
        
        # ========== BOTTLENECK ==========
        self.bottleneck1 = MultiStageConditionalBlock(512, 512, time_dim, stage_level=4)
        self.bottleneck2 = MultiStageConditionalBlock(512, 512, time_dim, stage_level=4)
        
        # ========== DECODER ==========
        # Level 4: 2x2 → 4x4
        self.up4 = nn.ConvTranspose2d(512, 512, 4, stride=2, padding=1)
        self.dec4 = MultiStageConditionalBlock(512 + 512, 256, time_dim, stage_level=3)
        
        # Level 3: 4x4 → 8x8
        self.up3 = nn.ConvTranspose2d(256, 256, 4, stride=2, padding=1)
        self.dec3 = MultiStageConditionalBlock(256 + 256, 128, time_dim, stage_level=2)
        
        # Level 2: 8x8 → 16x16
        self.up2 = nn.ConvTranspose2d(128, 128, 4, stride=2, padding=1)
        self.dec2 = MultiStageConditionalBlock(128 + 128, 64, time_dim, stage_level=1)
        
        # Level 1: 16x16 → 32x32
        self.up1 = nn.ConvTranspose2d(64, 64, 4, stride=2, padding=1)
        self.dec1 = MultiStageConditionalBlock(64 + 64, 64, time_dim, stage_level=1)
        
        # ========== OUTPUT ==========
        self.out = nn.Sequential(
            nn.GroupNorm(8, 64),
            nn.SiLU(),
            nn.Conv2d(64, 1, 3, padding=1)
        )
        
        # Initialize weights
        self.apply(self._init_weights)
        
    
    def _init_weights(self, module):
        if isinstance(module, (nn.Conv2d, nn.Linear, nn.ConvTranspose2d)):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.MultiheadAttention):
            nn.init.xavier_uniform_(module.in_proj_weight)
            nn.init.xavier_uniform_(module.out_proj.weight)
    
    def forward(self, x_noisy, tb, era5, t):
        """
        Forward pass with innovative conditioning:
        x_noisy: [N, 1, 32, 32] - noisy rainfall field
        tb: [N, 13, 32, 32] - brightness temperatures
        era5: [N, 5, 32, 32] - atmospheric variables
        t: [N] - timesteps
        """
        
        # ========== TIME EMBEDDING ==========
        t_emb = timestep_embedding(t, 256, device=x_noisy.device)
        t_emb = self.time_embed(t_emb)
        
        # ========== EXTRACT CONDITIONING FEATURES ==========
        # TB Encoder: Extract multi-scale TB features
        tb1, tb2, tb3, tb4 = self.tb_encoder(tb)
        # tb1: [N, 64, 16, 16] (after downsampling once)
        # tb2: [N, 128, 8, 8] (after downsampling twice)
        # tb3: [N, 256, 4, 4] (after downsampling three times)
        # tb4: [N, 512, 2, 2] (after downsampling four times)
        
        # ERA5 Processor: Convert to compact latent representation
        era5_latent = self.era5_processor(era5)  # [N, 512, 2, 2]
        
        # ========== NOISY IMAGE ENCODER WITH MULTI-STAGE INJECTION ==========
        # Initial projection (keep spatial dims 32x32)
        x = self.initial_proj(x_noisy)  # [N, 64, 32, 32]
        
        # Encoder block 1: 32x32 → 16x16 with TB injection
        x = self.enc1(x, t_emb)  # Process at 32x32
        skip1 = x  # [N, 64, 32, 32]
        x = self.down1(x)  # Downsample to 16x16
        
        # Inject TB features at 16x16 using cross-attention
        if tb1 is not None:
            x = x + tb1 + self.cross_attn_16(x, tb1)
        
        # Encoder block 2: 16x16 → 8x8 with TB injection
        x = self.enc2(x, t_emb)  # Process at 16x16
        skip2 = x  # [N, 128, 16, 16]
        x = self.down2(x)  # Downsample to 8x8
        
        # Inject TB features at 8x8 using cross-attention
        if tb2 is not None:
            x = x + tb2 + self.cross_attn_8(x, tb2)
        
        # Encoder block 3: 8x8 → 4x4 with TB injection
        x = self.enc3(x, t_emb)  
        skip3 = x  
        x = self.down3(x)  
        
        # Inject TB features at 4x4 using cross-attention
        if tb3 is not None:
            x = x + tb3 + self.cross_attn_4(x, tb3)
        
        # Encoder block 4: 4x4 → 2x2 with TB injection
        x = self.enc4(x, t_emb)  
        skip4 = x  
        x = self.down4(x)  
        

        fused_features = self.bottleneck_fusion(tb4, era5_latent)  # [N, 512, 2, 2]
        
        # Add fused features to bottleneck (with residual connection)
        x = x + fused_features
        
        # Bottleneck processing at 2x2
        x = self.bottleneck1(x, t_emb)
        x = self.bottleneck2(x, t_emb)
        

        # Decoder block 4: 2x2 → 4x4
        x = self.up4(x)  # Upsample to 4x4
        
        x = torch.cat([x, skip4], dim=1)  
        x = self.dec4(x, t_emb)
        

        x = self.up3(x)  
        
        x = torch.cat([x, skip3], dim=1)  
        x = self.dec3(x, t_emb)
        

        x = self.up2(x)  # Upsample to 16x16
        
        
        x = torch.cat([x, skip2], dim=1)  # Skip connection
        x = self.dec2(x, t_emb)
        
        # Decoder block 1: 16x16 → 32x32
        x = self.up1(x)  # Upsample to 32x32

        x = torch.cat([x, skip1], dim=1)  # Skip connection
        x = self.dec1(x, t_emb)
        
        # ========== Predicted Output Noise ==========
        predicted_noise = self.out(x)
        
        return predicted_noise

## 15. Diffusion schedule and sampling

Linear beta schedule and full DDPM reverse sampling. Guidance scale 1 uses conditioned predictions only. No conditioning-dropout training is supplied, so keep that sampling setting.

In [ ]:
class DiffusionProcess:
    """Simple diffusion process for signed residue."""
    def __init__(self, timesteps=500, device='cuda', zero_threshold=0.2):
        self.timesteps = timesteps
        self.device = device
        self.zero_threshold = zero_threshold
        
        # Linear noise schedule
        self.betas = torch.linspace(1e-4, 0.02, timesteps, device=device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
        
        # Calculations for posterior
        self.posterior_variance = (
            self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)
        )
        
        print(f"\nDiffusion Process:")
        print(f"  Timesteps: {timesteps}")
        print(f"  Zero threshold: {zero_threshold} mm/hr")
        print("  Clipping: disabled for inputs, diffusion states, sampling noise, and outputs")
    
    def add_noise(self, x_clean, t, processor):
        """Add noise directly to residue images without normalization."""
        x_clean_norm = processor.transform_rain_to_normalized(x_clean, device=self.device)
        
        noise = torch.randn_like(x_clean_norm)
        
        # Ensure t is properly shaped
        if t.dim() == 0:
            t = t.unsqueeze(0)
        
        sqrt_alpha = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        
        x_noisy = sqrt_alpha * x_clean_norm + sqrt_one_minus_alpha * noise
        
        return x_noisy, noise
    
    def sample(self, model, tb_features, era5_features, processor, batch_size=4, 
               guidance_scale=1.5, classifier_free_guidance=True,
               num_realizations=1):
        """Generate rain with ensemble sampling"""
        device = self.device
        
        print(f"\nSampling {num_realizations} rainfall realizations...")
        
        all_generated = []
        
        for k in range(num_realizations):
            print(f"\nGenerating realization {k+1}/{num_realizations}...")
            
            # Start from pure noise in residue space
            # Different noise for each realization
            torch.manual_seed(k * 1234)  # Different seed for each realization
            x = torch.randn(batch_size, 1, 32, 32, device=device)
            
            for t in tqdm(reversed(range(self.timesteps)), desc=f"Sampling {k+1}"):
                t_tensor = torch.full((batch_size,), t, device=device, dtype=torch.long)
                
                with torch.no_grad():
                    # Forward pass with conditioning
                    predicted_noise = model(x, tb_features[:batch_size], 
                                           era5_features[:batch_size], t_tensor)
                    
                    # Apply classifier-free guidance
                    if classifier_free_guidance and guidance_scale > 1.0:
                        # Unconditional prediction with zero conditioning
                        tb_zero = torch.zeros_like(tb_features[:batch_size])
                        era5_zero = torch.zeros_like(era5_features[:batch_size])
                        predicted_noise_uncond = model(x, tb_zero, era5_zero, t_tensor)
                        
                        # Linear combination
                        predicted_noise = predicted_noise_uncond + guidance_scale * (predicted_noise - predicted_noise_uncond)
                
                # Get parameters
                alpha = self.alphas[t]
                alpha_cumprod = self.alphas_cumprod[t]
                beta = self.betas[t]
                
                # DDPM sampling
                if t > 0:
                    noise = torch.randn_like(x)
                    model_variance = self.posterior_variance[t]
                    model_std = torch.sqrt(model_variance)
                else:
                    noise = 0
                    model_std = 0
                
                model_mean = (1 / torch.sqrt(alpha)) * (
                    x - beta * predicted_noise / torch.sqrt(1 - alpha_cumprod)
                )
                
                x = model_mean + model_std * noise
                
            # No inverse transform is needed for residue
            generated_physical = processor.transform_rain_to_physical(x)
            
            all_generated.append(generated_physical)
        
        # Stack all realizations
        # Always retain the realization axis, even for a single realization.
        all_generated = torch.stack(all_generated, dim=1)  # [B, R, 1, 32, 32]
        
        return all_generated

## 16. Masked mini-batch rainfall SWD

Both fields use the DPR > threshold mask. Dry locations become zeros in patch vectors; projections are freshly sampled on each call. Sorting is across patches for every direction.

In [ ]:
def minibatch_rainy_swd_squared(predicted_rain, true_rain, rain_threshold=0.2,
                               num_projections=32):
    """Squared image-level SWD with 32 random directions across a minibatch.

    Each masked rainfall patch is one H*W-dimensional observation. The same
    DPR > threshold mask is applied to prediction and target in each patch;
    this loss does not penalize predicted rain at DPR-dry locations. Random
    unit vectors project the patches to [batch, num_projections], which are
    sorted across patches independently for each direction. Units: (mm/hr)^2.
    """
    if predicted_rain.shape != true_rain.shape:
        raise ValueError(
            f"Shape mismatch: predicted_rain={predicted_rain.shape}, "
            f"true_rain={true_rain.shape}"
        )
    if predicted_rain.ndim != 4 or predicted_rain.shape[1] != 1:
        raise ValueError("Expected rainfall tensors with shape [batch, 1, height, width]")
    if num_projections < 1:
        raise ValueError("num_projections must be positive")
    rain_mask = true_rain > rain_threshold
    if not bool(rain_mask.any()):
        return predicted_rain.sum() * 0.0
    pred = (predicted_rain.float() * rain_mask).flatten(start_dim=1)
    target = (true_rain.float() * rain_mask).flatten(start_dim=1)
    directions = torch.randn(pred.shape[1], num_projections,
                             dtype=pred.dtype, device=pred.device)
    directions = F.normalize(directions, p=2, dim=0)
    pred_projected = pred @ directions
    target_projected = target @ directions
    return (pred_projected.sort(dim=0).values -
            target_projected.sort(dim=0).values).square().mean()

## 17. Balanced patch loader

Expected shape: (32,32,34). TB: 0–12; ERA5: 13,14,15,16,17,21,22,23,25; DPR: 31; ResUNet: 32; signed residual: 33. LSM is channel 25. Keep ≥10% nonzero residual pixels; reject bad shapes, NaNs, rainfall/prediction maxima >500, or inconsistent residuals. Scanning follows sorted filenames and stops at the two LSM quotas.

In [ ]:
def load_and_process_data(patches_dir, num_samples=55000, processor=None, fit_scalers=True, normalize_inputs=True):
    """Load and process data for residue-based diffusion training with balanced LSM sampling.

    Patch assumptions:
        - patch[:, :, :13]    -> TB channels
        - patch[:, :, [13,14,15,16,17,21,22,23,25]] -> selected ERA5 channels
        - patch[:, :, -3:-2]  -> true rainfall
        - patch[:, :, -2:-1]  -> U-Net prediction
        - patch[:, :, -1:]    -> true residue = true rainfall - U-Net prediction

    Selection rule added:
        - Keep only patches where at least 10% of pixels have nonzero residue
          either positive or negative.
    """

    import os
    import numpy as np
    import torch
    from tqdm import tqdm

    print(f"Loading balanced samples from {patches_dir} ...")
    print(f"Requested total samples: {num_samples}")

    all_files = [f for f in os.listdir(patches_dir) if f.endswith(".npy")]
    all_files = sorted(all_files)

    if len(all_files) == 0:
        raise ValueError(f"No .npy files found in {patches_dir}")

    target_per_case = num_samples // 2
    print(f"Target per LSM case: {target_per_case}")

    tb_case0, era5_case0, unet_case0, residue_case0, rain_case0 = [], [], [], [], []
    tb_case1, era5_case1, unet_case1, residue_case1, rain_case1 = [], [], [], [], []

    files_case0, files_case1 = [], []
    valid_case0 = 0
    valid_case1 = 0

    skipped_bad_shape = 0
    skipped_nan = 0
    skipped_extreme = 0
    skipped_inconsistent = 0
    skipped_low_residue_pixels = 0
    skipped_other = 0

    pbar = tqdm(all_files, desc="Scanning patches")

    for fname in pbar:
        if len(tb_case0) >= target_per_case and len(tb_case1) >= target_per_case:
            break

        try:
            patch_path = os.path.join(patches_dir, fname)
            patch = np.load(patch_path)

            if patch.shape != (32, 32, 34):
                skipped_bad_shape += 1
                continue

            tb = patch[:, :, :13]
            era5 = patch[:, :, [13, 14, 15, 16, 17, 21, 22, 23, 25]]

            true_rain = patch[:, :, -3:-2]
            unet_pred = patch[:, :, -2:-1]
            residue = patch[:, :, -1:]

            lsm = era5[:, :, -1]
            mean_lsm = np.mean(lsm)

            if (
                np.isnan(tb).any()
                or np.isnan(era5).any()
                or np.isnan(true_rain).any()
                or np.isnan(unet_pred).any()
                or np.isnan(residue).any()
                or np.isnan(lsm).any()
            ):
                skipped_nan += 1
                continue

            if true_rain.max() > 500 or unet_pred.max() > 500:
                skipped_extreme += 1
                continue

            reconstructed_residue = true_rain - unet_pred
            if not np.allclose(residue, reconstructed_residue, atol=1e-4):
                skipped_inconsistent += 1
                continue

            # ---------------------------------------------------------
            # Keep only patches where >=10% pixels have nonzero residue
            # Residue may be either positive or negative
            # ---------------------------------------------------------
            residue_2d = residue[:, :, 0]
            nonzero_residue_fraction = np.mean(np.abs(residue_2d) > 0)

            if nonzero_residue_fraction < MIN_NONZERO_RESIDUE_FRACTION:
                skipped_low_residue_pixels += 1
                continue

            # ---------------------------------------------------------
            # LSM rule
            # ---------------------------------------------------------
            if mean_lsm < 0.5:
                valid_case0 += 1
                if len(tb_case0) < target_per_case:
                    files_case0.append(fname)
                    tb_case0.append(tb)
                    era5_case0.append(era5)
                    unet_case0.append(unet_pred)
                    residue_case0.append(residue)
                    rain_case0.append(true_rain)
            else:
                valid_case1 += 1
                if len(tb_case1) < target_per_case:
                    files_case1.append(fname)
                    tb_case1.append(tb)
                    era5_case1.append(era5)
                    unet_case1.append(unet_pred)
                    residue_case1.append(residue)
                    rain_case1.append(true_rain)

            pbar.set_postfix({
                "saved_case0": len(tb_case0),
                "saved_case1": len(tb_case1),
                "valid_case0": valid_case0,
                "valid_case1": valid_case1,
                "low_residue": skipped_low_residue_pixels
            })

        except Exception:
            skipped_other += 1
            continue

    tb_list = tb_case0 + tb_case1
    era5_list = era5_case0 + era5_case1
    unet_list = unet_case0 + unet_case1
    residue_list = residue_case0 + residue_case1
    true_rain_list = rain_case0 + rain_case1

    loaded_case0 = len(tb_case0)
    loaded_case1 = len(tb_case1)

    print("\n" + "=" * 60)
    print("LSM BALANCING SUMMARY")
    print("=" * 60)
    print(f"Valid patches found for case 0 (mean_lsm < 0.5) : {valid_case0}")
    print(f"Valid patches found for case 1 (mean_lsm >= 0.5): {valid_case1}")
    print(f"Loaded patches from case 0                    : {loaded_case0}")
    print(f"Loaded patches from case 1                    : {loaded_case1}")
    print(f"Total loaded                                  : {loaded_case0 + loaded_case1}")

    if loaded_case0 < target_per_case or loaded_case1 < target_per_case:
        print("\nWarning: could not reach fully balanced target.")
        print(f"Requested per case: {target_per_case}")
        print(f"Obtained case 0  : {loaded_case0}")
        print(f"Obtained case 1  : {loaded_case1}")

    print("\nSkipped patches summary:")
    print(f"  Bad shape          : {skipped_bad_shape}")
    print(f"  NaN                : {skipped_nan}")
    print(f"  Extreme values     : {skipped_extreme}")
    print(f"  Bad residue        : {skipped_inconsistent}")
    print(f"  Low residue pixels : {skipped_low_residue_pixels}")
    print(f"  Other errors       : {skipped_other}")

    if len(tb_list) == 0:
        raise ValueError("No valid balanced patches found!")

    indices = np.arange(len(tb_list))
    np.random.shuffle(indices)

    processor.loaded_files = [(files_case0 + files_case1)[int(i)] for i in indices]
    tb_array = np.array(tb_list, dtype=np.float32)[indices]
    era5_array = np.array(era5_list, dtype=np.float32)[indices]
    unet_array = np.array(unet_list, dtype=np.float32)[indices]
    residue_array = np.array(residue_list, dtype=np.float32)[indices]
    true_rain_array = np.array(true_rain_list, dtype=np.float32)[indices]

    print(f"\nLoaded {len(tb_array)} balanced samples")
    print(f"TB shape         : {tb_array.shape}")
    print(f"ERA5 shape       : {era5_array.shape}")
    print(f"U-Net pred shape : {unet_array.shape}")
    print(f"Residue shape    : {residue_array.shape}")
    print(f"True rain shape  : {true_rain_array.shape}")

    residue_flat = residue_array.flatten()
    residue_flat = residue_flat[~np.isnan(residue_flat)]

    print("\nResidue distribution:")
    print(f"  Mean: {residue_flat.mean():.4f}")
    print(f"  Std : {residue_flat.std():.4f}")
    print(f"  Min : {residue_flat.min():.4f}")
    print(f"  Max : {residue_flat.max():.4f}")
    print(f"  < 0 : {(residue_flat < 0).sum()} ({(residue_flat < 0).mean():.2%})")
    print(f"  = 0 : {(residue_flat == 0).sum()} ({(residue_flat == 0).mean():.2%})")
    print(f"  > 0 : {(residue_flat > 0).sum()} ({(residue_flat > 0).mean():.2%})")

    rain_flat = true_rain_array.flatten()
    rain_flat = rain_flat[~np.isnan(rain_flat)]

    print("\nTrue rainfall distribution:")
    print(f"  Zero values: {(rain_flat == 0).sum()} ({(rain_flat == 0).mean():.2%})")
    print(f"  >0 mm/hr   : {(rain_flat > 0).sum()} ({(rain_flat > 0).mean():.2%})")

    if (rain_flat > 0).any():
        nonzero = rain_flat[rain_flat > 0]
        print(f"  Non-zero mean    : {nonzero.mean():.4f} mm/hr")
        print(f"  Non-zero max     : {nonzero.max():.4f} mm/hr")
        print(f"  95th percentile : {np.percentile(nonzero, 95):.4f} mm/hr")
        print(f"  99th percentile : {np.percentile(nonzero, 99):.4f} mm/hr")

    if processor is None:        processor = SimpleDataProcessor()

    if fit_scalers:
        print("\nFitting processor...")
        tb_stats, era5_stats = processor.fit(tb_array, era5_array, residue_array)
    else:
        tb_stats, era5_stats = None, None

    print("Transforming data...")
    tb_transformed = processor.transform_tb(tb_array) if normalize_inputs else tb_array
    era5_transformed = processor.transform_era5(era5_array) if normalize_inputs else era5_array

    tb_tensor = torch.FloatTensor(tb_transformed).permute(0, 3, 1, 2)
    era5_tensor = torch.FloatTensor(era5_transformed).permute(0, 3, 1, 2)
    unet_tensor = torch.FloatTensor(unet_array).permute(0, 3, 1, 2)
    residue_tensor = torch.FloatTensor(residue_array).permute(0, 3, 1, 2)
    true_rain_tensor = torch.FloatTensor(true_rain_array).permute(0, 3, 1, 2)

    return (
        tb_tensor,
        era5_tensor,
        unet_tensor,
        residue_tensor,
        true_rain_tensor,
        processor,
        tb_stats,
        era5_stats
    )

## 18. Colormap and comparison helpers

Original comparison functions, with one shared symmetric difference scale in the three-row figure.

In [ ]:
def create_custom_colormap():
    """Create a custom colormap with white for values below threshold"""
    import matplotlib.colors as mcolors
    import numpy as np
    
    # Define the colors
    colors = [
        (1.0, 1.0, 1.0, 0.0),  # White with 0 alpha for values < 0.2
        (1.0, 1.0, 1.0, 1.0),  # White for 0.2
        (0.9, 0.9, 0.0, 1.0),  # Yellow for moderate values
        (1.0, 0.5, 0.0, 1.0),  # Orange
        (1.0, 0.0, 0.0, 1.0),  # Red for high values
        (0.5, 0.0, 0.0, 1.0)   # Dark red for very high values
    ]
    
    # Create colormap
    return mcolors.LinearSegmentedColormap.from_list('rain_cmap', colors)

def plot_2x5_comparison(ground_truth, generated_ensemble, zero_threshold=0.2, num_samples=5):
    """
    Create a 2x5 comparison plot:
    Top row: Ground truth for 5 samples
    Bottom row: Average of 5 generated realizations for each sample
    """
    
    # Keep values as-is; no zero-threshold clipping
    generated_np = generated_ensemble.cpu().numpy()
    ground_truth_np = ground_truth.cpu().numpy()
    
    # Take average over the 5 realizations
    generated_avg = np.mean(generated_np, axis=1)  # Average over realizations dimension
    
    # Create custom colormap
    rain_cmap = create_custom_colormap()
    
    # Find global max for consistent color scaling
    vmax = max(ground_truth_np.max(), generated_avg.max())
    vmax = max(vmax, 5.0)  # At least show up to 5 mm/hr
    
    # Create figure
    fig, axes = plt.subplots(2, num_samples, figsize=(20, 8))
    
    # Plot ground truth (top row)
    for i in range(num_samples):
        im1 = axes[0, i].imshow(ground_truth_np[i, 0], cmap=rain_cmap, vmin=0, vmax=vmax)
        axes[0, i].set_title(f'Sample {i+1}\nGround Truth', fontsize=12, fontweight='bold')
        axes[0, i].axis('off')
        
        # Add text with statistics
        gt_data = ground_truth_np[i, 0]
        zero_frac = (gt_data < zero_threshold).mean()
        mean_val = gt_data[gt_data >= zero_threshold].mean() if (gt_data >= zero_threshold).any() else 0
        max_val = gt_data.max()
        axes[0, i].text(0.5, -0.15, f'Mean: {mean_val:.2f} | Max: {max_val:.2f}\nZero: {zero_frac:.0%}', 
                       transform=axes[0, i].transAxes, ha='center', fontsize=9)
    
    # Plot generated average (bottom row)
    for i in range(num_samples):
        im2 = axes[1, i].imshow(generated_avg[i, 0], cmap=rain_cmap, vmin=0, vmax=vmax)
        axes[1, i].set_title(f'Sample {i+1}\nGenerated (Avg of 5)', fontsize=12, fontweight='bold')
        axes[1, i].axis('off')
        
        # Add text with statistics
        gen_data = generated_avg[i, 0]
        zero_frac = (gen_data < zero_threshold).mean()
        mean_val = gen_data[gen_data >= zero_threshold].mean() if (gen_data >= zero_threshold).any() else 0
        max_val = gen_data.max()
        axes[1, i].text(0.5, -0.15, f'Mean: {mean_val:.2f} | Max: {max_val:.2f}\nZero: {zero_frac:.0%}', 
                       transform=axes[1, i].transAxes, ha='center', fontsize=9)
    
    # Add colorbar
    plt.subplots_adjust(right=0.85)
    cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im2, cax=cbar_ax)
    cbar.set_label('Rain Rate (mm/hr)', fontsize=12)
    
    # Add title
    plt.suptitle(f'Rainfall Estimation Comparison (Zero Threshold: {zero_threshold} mm/hr)', 
                fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'plot_2x5_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return generated_avg


def plot_3x5_comparison(ground_truth, generated_ensemble, zero_threshold=0.2, num_samples=5, epoch=None):
    """
    Create a 3x5 comparison plot with your specified style:
    Row 1: Ground truth for 5 samples
    Row 2: Average of 5 generated realizations for each sample
    Row 3: Difference (Generated - True)
    """
    
    # Keep values as-is; no zero-threshold clipping
    generated_np = generated_ensemble.cpu().numpy()
    ground_truth_np = ground_truth.detach().cpu().numpy().copy()
    if generated_np.ndim != 5 or ground_truth_np.ndim != 4:
        raise ValueError(f"Expected ensemble [B,R,1,H,W] and truth [B,1,H,W]; got {generated_np.shape} and {ground_truth_np.shape}")
    if generated_np.shape[0] != ground_truth_np.shape[0]:
        raise ValueError("Ensemble and truth batch dimensions differ")
    generated_avg = np.mean(generated_np, axis=1)  # [B,1,H,W]
    
    # Create figure
    fig, axes = plt.subplots(3, num_samples, figsize=(4*num_samples, 12))
    
    if num_samples == 1:
        axes = axes.reshape(3, 1)
    
    # Create title
    title = 'True vs Generated Rain (mm/hr)' if not epoch else f'Epoch {epoch} - True vs Generated Rain (mm/hr)'
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    # Find reasonable color scale for rain maps
    all_data = np.concatenate([ground_truth_np[:num_samples], generated_avg[:num_samples]])
    if (all_data > 0).any():
        rain_nonzero = all_data[all_data > 0]
        vmax = np.percentile(rain_nonzero, 95) if len(rain_nonzero) > 0 else 1.0
        vmax = max(vmax, 1.0)
    else:
        vmax = 1.0
    
    shared_diff_limit = max(float(np.max(np.abs(generated_avg[:num_samples] - ground_truth_np[:num_samples]))), 0.5)
    for i in range(num_samples):
        # Row 1: True rain
        ax_true = axes[0, i]
        im_true = ax_true.imshow(ground_truth_np[i, 0], cmap='YlGnBu', vmin=0, vmax=vmax)
        ax_true.set_title(f'True Sample {i+1}', fontsize=10)
        ax_true.axis('off')
        
        true_sample = ground_truth_np[i, 0]
        ax_true.text(0.02, 0.98, f'Zeros: {(true_sample == 0).mean():.0%}\nMax: {true_sample.max():.2f}',
                    transform=ax_true.transAxes, fontsize=8,
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Row 2: Generated rain (average of realizations)
        ax_gen = axes[1, i]
        im_gen = ax_gen.imshow(generated_avg[i, 0], cmap='YlGnBu', vmin=0, vmax=vmax)
        ax_gen.set_title(f'Generated Sample {i+1}\n(Avg of {generated_np.shape[1]})', fontsize=10)
        ax_gen.axis('off')
        
        gen_sample = generated_avg[i, 0]
        ax_gen.text(0.02, 0.98, f'Zeros: {(gen_sample == 0).mean():.0%}\nMax: {gen_sample.max():.2f}',
                   transform=ax_gen.transAxes, fontsize=8,
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Row 3: Difference (Generated - True)
        ax_diff = axes[2, i]
        diff = generated_avg[i, 0] - ground_truth_np[i, 0]
        
        # Calculate symmetric color limits for difference
        max_abs_diff = np.max(np.abs(diff)) if len(diff) > 0 else 0.5
        max_abs_diff = shared_diff_limit
        
        im_diff = ax_diff.imshow(diff, cmap='RdBu_r', vmin=-max_abs_diff, vmax=max_abs_diff)
        ax_diff.set_title(f'Difference {i+1}', fontsize=10)
        ax_diff.axis('off')
        
        # Calculate statistics for difference
        sample_mae = np.mean(np.abs(diff))
        sample_rmse = np.sqrt(np.mean(diff**2))
        
        ax_diff.text(0.02, 0.98, f'MAE: {sample_mae:.3f}\nRMSE: {sample_rmse:.3f}',
                    transform=ax_diff.transAxes, fontsize=8,
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    
    # Add colorbars
    if num_samples > 0:
        plt.subplots_adjust(right=0.85)
        
        # Colorbar for rain maps (top)
        cbar_ax1 = fig.add_axes([0.88, 0.68, 0.02, 0.25])
        cbar1 = plt.colorbar(im_true, cax=cbar_ax1)
        cbar1.set_label('Rain Rate (mm/hr)', fontsize=10)
        
        # Colorbar for difference maps (bottom)
        cbar_ax2 = fig.add_axes([0.88, 0.15, 0.02, 0.25])
        cbar2 = plt.colorbar(im_diff, cax=cbar_ax2)
        cbar2.set_label('Difference (mm/hr)', fontsize=10)
    
    fig.savefig(OUTPUT_DIR / 'plot_3x5_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print overall statistics
    print("\n" + "="*60)
    print("OVERALL STATISTICS FOR ALL SAMPLES")
    print("="*60)
    
    mae_values = []
    rmse_values = []
    
    for i in range(num_samples):
        diff = generated_avg[i, 0] - ground_truth_np[i, 0]
        mae = np.mean(np.abs(diff))
        rmse = np.sqrt(np.mean(diff**2))
        mae_values.append(mae)
        rmse_values.append(rmse)
        
        print(f"\nSample {i+1}:")
        print(f"  Ground Truth: Zeros={(ground_truth_np[i, 0] == 0).mean():.1%}, Max={ground_truth_np[i, 0].max():.3f}")
        print(f"  Generated:    Zeros={(generated_avg[i, 0] == 0).mean():.1%}, Max={generated_avg[i, 0].max():.3f}")
        print(f"  Difference:   MAE={mae:.4f}, RMSE={rmse:.4f}")
    
    print(f"\nAverage across {num_samples} samples:")
    print(f"  MAE: {np.mean(mae_values):.4f} ± {np.std(mae_values):.4f}")
    print(f"  RMSE: {np.mean(rmse_values):.4f} ± {np.std(rmse_values):.4f}")
    
    return generated_avg


## 19. Rainfall diagnostics and residual plots

Inspect negative outputs and ensemble statistics, then compare true and mean generated residuals.

In [ ]:
def print_rainfall_diagnostics(true_rain, generated_ensemble):
    """Compare paired DPR and generated fields without clipping either one.

    true_rain: [B,1,H,W]; generated_ensemble: [B,R,1,H,W].
    For R>1, compare DPR against the ensemble mean, and also print member statistics.
    """
    if true_rain.ndim != 4 or generated_ensemble.ndim != 5:
        raise ValueError("Expected DPR [B,1,H,W] and ensemble [B,R,1,H,W]")
    if generated_ensemble.shape[0] != true_rain.shape[0] or generated_ensemble.shape[2:] != true_rain.shape[1:]:
        raise ValueError("DPR and generated rainfall shapes do not match")

    truth = true_rain.detach().float().cpu().numpy()[:, 0]
    members = generated_ensemble.detach().float().cpu().numpy()[:, :, 0]
    pred = members.mean(axis=1)
    thresholds = (0.2, 1.0, 8.0, 16.0)
    percentiles = (50, 90, 95, 99, 99.9)

    def describe(name, field):
        values = field.reshape(-1)
        p = np.percentile(values, percentiles)
        positive = values[values > 0.2]
        print(f"  {name:16s} min={values.min():9.3f} mean={values.mean():9.3f} max={values.max():9.3f} mm/hr")
        print("    all-pixel percentiles " + ", ".join(f"P{q:g}={v:.3f}" for q, v in zip(percentiles, p)))
        if positive.size:
            q = np.percentile(positive, (50, 90, 99, 99.9))
            print("    >0.2 percentiles     " + ", ".join(f"P{k:g}={v:.3f}" for k, v in zip((50,90,99,99.9), q)))
        print("    pixel fractions       " + ", ".join(f">{t:g}={np.mean(values > t):.2%}" for t in thresholds))
        print(f"    negative={np.mean(values < 0):.2%}; exact zero={np.mean(values == 0):.2%}")

    print("\n" + "=" * 72)
    print(f"PAIRED RAINFALL DIAGNOSTICS: {len(truth)} validation patches, {members.shape[1]} member(s)")
    print("All rainfall values below are raw; no clipping or threshold is applied.")
    describe("DPR", truth)
    describe("Generated mean", pred)
    if members.shape[1] > 1:
        describe("All members", members)

    for t in thresholds:
        observed = truth > t
        forecast = pred > t
        hits = np.count_nonzero(observed & forecast)
        false = np.count_nonzero(~observed & forecast)
        missed = np.count_nonzero(observed & ~forecast)
        print(f"  threshold >{t:g}: hits={hits}, false alarms={false}, misses={missed}, "
              f"false-alarm pixel fraction={false / observed.size:.2%}, "
              f"frequency bias={(hits + false) / (hits + missed):.3f}" if hits + missed else
              f"  threshold >{t:g}: no observed rainy pixels")
    for i, (y, x) in enumerate(zip(truth, pred), 1):
        print(f"  patch {i}: DPR max={y.max():.3f}, generated max={x.max():.3f}; "
              f"DPR >0.2={(y > 0.2).mean():.2%}, generated >0.2={(x > 0.2).mean():.2%}; "
              f"MAE={np.mean(np.abs(x-y)):.3f}")
    print("=" * 72)

def plot_true_vs_predicted_residue(true_residue, predicted_residue, num_samples=5):
    true_np = true_residue.detach().cpu().numpy() if torch.is_tensor(true_residue) else true_residue
    pred_np = predicted_residue.detach().cpu().numpy() if torch.is_tensor(predicted_residue) else predicted_residue

    # Convert to shape: (N, H, W)
    if true_np.ndim == 4:
        true_np = true_np[:, 0]
    if pred_np.ndim == 4:
        pred_np = pred_np[:, 0]

    n = min(num_samples, true_np.shape[0], pred_np.shape[0])

    if true_np.ndim != 3 or pred_np.ndim != 3 or true_np.shape[1:] != pred_np.shape[1:]:
        raise ValueError(f"Expected matching [B,H,W] residue maps; got {true_np.shape} and {pred_np.shape}")
    if n == 0:
        raise ValueError("No residue samples to plot")
    vmax = max(float(np.nanmax(np.abs(true_np[:n]))),
               float(np.nanmax(np.abs(pred_np[:n]))), 1e-6)

    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))

    if n == 1:
        axes = np.expand_dims(axes, axis=1)

    for i in range(n):
        im1 = axes[0, i].imshow(true_np[i], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[0, i].set_title(f"True Residue {i+1}")
        axes[0, i].axis("off")

        im2 = axes[1, i].imshow(pred_np[i], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[1, i].set_title(f"Predicted Residue {i+1}")
        axes[1, i].axis("off")

    fig.colorbar(im2, ax=axes.ravel().tolist(), shrink=0.8, label="Residue (mm/hr)")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'residual_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 20. Load selected patches without normalization

Keep raw input tensors until the train/validation split is known. This corrects the source’s fitting of scalers before splitting.

In [ ]:
if not PATCHES_DIR.is_dir():
    raise FileNotFoundError(f'Place patches at {PATCHES_DIR}, or edit cell 4.')
processor = SimpleDataProcessor(zero_threshold=ZERO_THRESHOLD)
(tb_all, era5_all, unet_all, residue_all, true_rain_all,
 processor, tb_stats, era5_stats) = load_and_process_data(
    str(PATCHES_DIR), num_samples=NUM_SAMPLES, processor=processor,
    fit_scalers=False, normalize_inputs=False,
)

## 21. Split, fit scalers, and create loaders

There is no test split in the supplied workflow. Save filenames for the selected training and validation patches. In sample-only mode, reuse checkpoint scalers instead of fitting new ones. Load only checkpoints you trust: the original checkpoint format contains serialized NumPy/scaler state.

In [ ]:
dataset = TensorDataset(tb_all, era5_all, unet_all, residue_all, true_rain_all)
train_size = int(TRAIN_FRACTION * len(dataset))
val_size = len(dataset) - train_size
if min(train_size, val_size) < 1:
    raise ValueError('Need at least one sample in each split.')
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
train_indices = train_dataset.indices
val_indices = val_dataset.indices
if SAMPLE_ONLY:
    if not BEST_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(BEST_CHECKPOINT_PATH)
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    state = checkpoint['processor_state']
    for scaler, saved in zip(processor.tb_scalers, state['tb_scalers']):
        scaler.__dict__.update(saved)
    for scaler, saved in zip(processor.era5_scalers, state['era5_scalers']):
        scaler.__dict__.update(saved)
    processor.rain_stats = state['rain_stats']
    processor.zero_threshold = state['zero_threshold']
    zero_threshold = processor.zero_threshold
else:
    processor.fit(tb_all[train_indices].permute(0,2,3,1).numpy(),
                  era5_all[train_indices].permute(0,2,3,1).numpy(),
                  residue_all[train_indices].numpy())
# Apply channel transforms in-place without allocating another full normalized dataset.
for tensor, scalers in [(tb_all, processor.tb_scalers), (era5_all, processor.era5_scalers)]:
    for channel, scaler in enumerate(scalers):
        tensor[:, channel].sub_(float(scaler.mean_[0])).div_(float(scaler.scale_[0]))
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False, drop_last=False)
with open(OUTPUT_DIR / 'selected_file_splits.json', 'w') as f:
    json.dump({'seed': SEED,
               'train': [processor.loaded_files[i] for i in train_indices],
               'validation': [processor.loaded_files[i] for i in val_indices]}, f, indent=2)
print('Train:', train_size, '| validation:', val_size)

## 22. Initialize model and diffusion

The forward check verifies tensor shapes before optimization.

In [ ]:
model = DiffusionRainModel().to(device)
diffusion = DiffusionProcess(timesteps=DIFFUSION_STEPS, device=device, zero_threshold=zero_threshold)
print('Model parameters:', sum(p.numel() for p in model.parameters()))
with torch.no_grad():
    test_output = model(torch.randn(2,1,32,32,device=device),
                        torch.randn(2,13,32,32,device=device),
                        torch.randn(2,9,32,32,device=device),
                        torch.randint(0, DIFFUSION_STEPS, (2,), device=device))
assert test_output.shape == (2,1,32,32)
print('Forward shape check passed')

## 23. Optimizer and history

Re-running this cell resets the optimizer and loss history. SAMPLE_ONLY performs zero training epochs.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
num_epochs = 0 if SAMPLE_ONLY else NUM_EPOCHS
train_losses, val_losses = [], []
train_mse_losses, train_swd_losses = [], []
val_mse_losses, val_swd_losses = [], []
best_val_loss = float('inf')

## 24. Train and validate

Retains the two forward passes and original means of batch losses, including smaller final batches. Validation noise, timesteps, and projection directions are resampled each epoch; validation loss is stochastic. Checkpoints track the lowest weighted validation loss.

In [ ]:
for epoch in range(num_epochs):
    model.train()

    epoch_train_loss = 0.0
    epoch_mse_loss = 0.0
    epoch_swd_loss = 0.0

    for batch_idx, (tb_batch, era5_batch, unet_batch, residue_batch, true_rain_batch) in enumerate(
        tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    ):
        tb_batch = tb_batch.to(device)
        era5_batch = era5_batch.to(device)
        unet_batch = unet_batch.to(device)
        residue_batch = residue_batch.to(device)
        true_rain_batch = true_rain_batch.to(device)

        # Standard DDPM MSE: draw one full-range timestep per sample.
        t_mse = torch.randint(0, diffusion.timesteps, (tb_batch.shape[0],), device=device)
        x_mse, true_noise = diffusion.add_noise(residue_batch, t_mse, processor)
        predicted_noise_mse = model(x_mse, tb_batch, era5_batch, t_mse)
        mse_loss = F.mse_loss(predicted_noise_mse, true_noise)

        # Rainfall SWD: one separate low timestep shared by the minibatch.
        t_swd = torch.randint(0, SWD_TIMESTEP_UPPER, (1,), device=device).expand(tb_batch.shape[0])
        x_swd, _ = diffusion.add_noise(residue_batch, t_swd, processor)
        predicted_noise_swd = model(x_swd, tb_batch, era5_batch, t_swd)

        # Reconstruct the implied clean residue r0 from r_t and epsilon_theta.
        sqrt_alpha_bar = diffusion.sqrt_alphas_cumprod[t_swd].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar = diffusion.sqrt_one_minus_alphas_cumprod[t_swd].view(-1, 1, 1, 1)
        predicted_residue = (
            x_swd - sqrt_one_minus_alpha_bar * predicted_noise_swd
        ) / (sqrt_alpha_bar + 1e-8)

        # Convert the predicted clean residue to physical rainfall.
        # Preserve the signed rainfall estimate during training so gradients
        # remain available for negative predictions.
        predicted_rain = torch.where(unet_batch > 0, unet_batch + predicted_residue, unet_batch)

        # Compare DPR-masked rainfall patches across 32 random projections.
        swd_loss = minibatch_rainy_swd_squared(
            predicted_rain,
            true_rain_batch,
            rain_threshold=zero_threshold,
            num_projections=NUM_PROJECTIONS,
        )

        loss = mse_weight * mse_loss + rain_swd_weight * swd_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()
        epoch_mse_loss += mse_loss.item()
        epoch_swd_loss += swd_loss.item()

    scheduler.step()

    avg_train_loss = epoch_train_loss / len(train_loader)
    avg_mse_loss = epoch_mse_loss / len(train_loader)
    avg_swd_loss = epoch_swd_loss / len(train_loader)

    train_losses.append(avg_train_loss)
    train_mse_losses.append(avg_mse_loss)
    train_swd_losses.append(avg_swd_loss)

    # =====================================================
    # Validation
    # =====================================================
    model.eval()

    epoch_val_loss = 0.0
    epoch_val_mse = 0.0
    epoch_val_swd = 0.0

    with torch.no_grad():
        for tb_batch, era5_batch, unet_batch, residue_batch, true_rain_batch in val_loader:
            tb_batch = tb_batch.to(device)
            era5_batch = era5_batch.to(device)
            unet_batch = unet_batch.to(device)
            residue_batch = residue_batch.to(device)
            true_rain_batch = true_rain_batch.to(device)

            t_mse = torch.randint(0, diffusion.timesteps, (tb_batch.shape[0],), device=device)
            x_mse, true_noise = diffusion.add_noise(residue_batch, t_mse, processor)
            predicted_noise_mse = model(x_mse, tb_batch, era5_batch, t_mse)
            val_mse = F.mse_loss(predicted_noise_mse, true_noise)

            t_swd = torch.randint(0, SWD_TIMESTEP_UPPER, (1,), device=device).expand(tb_batch.shape[0])
            x_swd, _ = diffusion.add_noise(residue_batch, t_swd, processor)
            predicted_noise_swd = model(x_swd, tb_batch, era5_batch, t_swd)

            sqrt_alpha_bar = diffusion.sqrt_alphas_cumprod[t_swd].view(-1, 1, 1, 1)
            sqrt_one_minus_alpha_bar = diffusion.sqrt_one_minus_alphas_cumprod[t_swd].view(-1, 1, 1, 1)
            predicted_residue = (
                x_swd - sqrt_one_minus_alpha_bar * predicted_noise_swd
            ) / (sqrt_alpha_bar + 1e-8)
            predicted_rain = torch.where(unet_batch > 0, unet_batch + predicted_residue, unet_batch)

            val_swd = minibatch_rainy_swd_squared(
                predicted_rain,
                true_rain_batch,
                rain_threshold=zero_threshold,
                num_projections=NUM_PROJECTIONS,
            )

            val_loss = mse_weight * val_mse + rain_swd_weight * val_swd

            epoch_val_loss += val_loss.item()
            epoch_val_mse += val_mse.item()
            epoch_val_swd += val_swd.item()

    avg_val_loss = epoch_val_loss / len(val_loader)
    avg_val_mse = epoch_val_mse / len(val_loader)
    avg_val_swd = epoch_val_swd / len(val_loader)

    val_losses.append(avg_val_loss)
    val_mse_losses.append(avg_val_mse)
    val_swd_losses.append(avg_val_swd)

    print(
        f"Epoch {epoch+1:3d}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.6f} (MSE: {avg_mse_loss:.6f}, RainSWD: {avg_swd_loss:.6f}) | "
        f"Val Loss: {avg_val_loss:.6f} (MSE: {avg_val_mse:.6f}, RainSWD: {avg_val_swd:.6f}) | "
        f"LR: {optimizer.param_groups[0]['lr']:.2e}"
    )

    # =====================================================
    # Save best model
    # =====================================================
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        checkpoint = {
            'epoch': epoch,
            'run_config': {'diffusion_steps': DIFFUSION_STEPS, 'swd_timestep_upper': SWD_TIMESTEP_UPPER, 'num_projections': NUM_PROJECTIONS, 'seed': SEED},
            'model_state_dict': copy.deepcopy(model.state_dict()),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'train_mse': avg_mse_loss,
            'train_swd': avg_swd_loss,
            'val_mse': avg_val_mse,
            'val_swd': avg_val_swd,
            'loss_weights': {
                'mse_weight': mse_weight,
                'rain_swd_weight': rain_swd_weight,
            },
            'processor_state': {
                'tb_scalers': [scaler.__dict__ for scaler in processor.tb_scalers],
                'era5_scalers': [scaler.__dict__ for scaler in processor.era5_scalers],
                'rain_stats': processor.rain_stats,
                'zero_threshold': processor.zero_threshold,
            },
            'zero_threshold': processor.zero_threshold,
        }

        torch.save(checkpoint, best_checkpoint_path)
        print(f"  ✓ Saved best model (val_loss: {best_val_loss:.6f})")

## 25. Plot and save loss history

Raw loss terms, weighted components, and total loss. In sample-only mode the history is empty.

In [ ]:
history = dict(train_loss=train_losses, val_loss=val_losses,
               train_mse=train_mse_losses, val_mse=val_mse_losses,
               train_swd=train_swd_losses, val_swd=val_swd_losses)
if train_losses:
    with open(OUTPUT_DIR / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    epochs = np.arange(1, len(train_losses)+1)
    fig, axes = plt.subplots(1,2,figsize=(13,4.5))
    for ax, tr, va, title in [(axes[0],train_mse_losses,val_mse_losses,'Noise MSE'),
                              (axes[1],train_swd_losses,val_swd_losses,'Rainfall SWD squared')]:
        ax.plot(epochs,tr,label='Train'); ax.plot(epochs,va,label='Validation')
        ax.set(title=title,xlabel='Epoch',ylabel=title)
        ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'raw_loss_terms.png',dpi=150,bbox_inches='tight'); plt.show()
    fig, ax = plt.subplots(figsize=(9,5))
    for values, weight, label in [(train_mse_losses,MSE_WEIGHT,'Train weighted MSE'),
                                  (val_mse_losses,MSE_WEIGHT,'Validation weighted MSE'),
                                  (train_swd_losses,RAIN_SWD_WEIGHT,'Train weighted SWD'),
                                  (val_swd_losses,RAIN_SWD_WEIGHT,'Validation weighted SWD'),
                                  (train_losses,1,'Train total'),(val_losses,1,'Validation total')]:
        ax.plot(epochs,np.array(values)*weight,label=label)
    ax.set(xlabel='Epoch',ylabel='Weighted loss',title='Loss components and total')
    ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'weighted_loss_history.png',dpi=150,bbox_inches='tight'); plt.show()
else:
    print('No optimization history in sample-only mode.')

## 26. Reload best model and select visualization patches

Use the best model and verify the sampling schedule against stored configuration. This is a validation preview, not independent test evaluation.

In [ ]:
checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device, weights_only=False)
saved_steps = checkpoint.get('run_config', {}).get('diffusion_steps', 1000)
if saved_steps != DIFFUSION_STEPS:
    raise ValueError(f'Checkpoint uses {saved_steps} diffusion steps; update cell 4.')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
val_tb, val_era5, val_unet, val_residue, val_true_rain = next(iter(val_loader))
num_visualization_samples = min(NUM_VISUALIZATION_SAMPLES, val_tb.shape[0])
val_tb, val_era5, val_unet, val_residue, val_true_rain = [
    x[:num_visualization_samples].to(device) for x in
    (val_tb, val_era5, val_unet, val_residue, val_true_rain)]
print('Best epoch:', checkpoint['epoch'] + 1, '| preview patches:', num_visualization_samples)

## 27. Sample residual ensemble and reconstruct rainfall

Residual corrections are added only where ResUNet > 0, matching the training rainfall reconstruction. Shape is [patch, realization, channel, height, width]. Sampling uses all reverse steps; extra realizations increase work.

In [ ]:
with torch.no_grad():
    generated_residue_ensemble = diffusion.sample(
        model, val_tb, val_era5, processor, batch_size=num_visualization_samples,
        guidance_scale=1, classifier_free_guidance=False, num_realizations=NUM_REALIZATIONS)
assert generated_residue_ensemble.shape[:2] == (num_visualization_samples, NUM_REALIZATIONS)
final_generated_ensemble = torch.where(
    val_unet.unsqueeze(1) > 0,
    val_unet.unsqueeze(1) + generated_residue_ensemble,
    val_unet.unsqueeze(1))
predicted_residue = generated_residue_ensemble.mean(dim=1)
print_rainfall_diagnostics(val_true_rain, final_generated_ensemble)
np.savez_compressed(OUTPUT_DIR / 'validation_preview_ensemble.npz',
    generated_residue=generated_residue_ensemble.cpu().numpy(),
    generated_rainfall=final_generated_ensemble.cpu().numpy(),
    true_rainfall=val_true_rain.cpu().numpy(),
    resunet_rainfall=val_unet.cpu().numpy(),
    true_residue=val_residue.cpu().numpy())

## 28. Compare residuals and rainfall

Display and save the original comparison figures. Rainfall errors are based on ensemble means, across all pixels of these preview patches.

In [ ]:
plot_true_vs_predicted_residue(val_residue, predicted_residue, num_samples=num_visualization_samples)
generated_avg = plot_3x5_comparison(
    val_true_rain, final_generated_ensemble, zero_threshold=zero_threshold,
    num_samples=num_visualization_samples, epoch=checkpoint['epoch']+1)


## 29. Ensemble variability and output summary

With one realization, variability across members is zero. Increase NUM_REALIZATIONS to inspect ensemble spread. The threshold fraction is not necessarily the exact zero fraction.

In [ ]:
for i in range(min(2, num_visualization_samples)):
    members = final_generated_ensemble[i,:,0].cpu().numpy()
    means = members.mean(axis=(1,2))
    maxima = members.max(axis=(1,2))
    below = (members < zero_threshold).mean(axis=(1,2))
    print(f'Patch {i+1}: across {NUM_REALIZATIONS} members')
    print(f'  Mean rain: {means.mean():.4f} ± {means.std():.4f} mm/hr')
    print(f'  Maximum: {maxima.mean():.4f} ± {maxima.std():.4f} mm/hr')
    print(f'  Below threshold: {below.mean():.2%} ± {below.std():.2%}')
print('Saved outputs:', OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file(): print(path.name)